# APK 6725 - AI for Sport & Movement Sciences
## Assignment 9: Convolutional Neural Networks for Activity Recognition

This assignment practices the ideas from lectures 05.01 to 05.02c. You will use 1D convolutional neural networks (CNNs) to classify daily activities from the **raw smartphone signals** of the UCI HAR dataset. In Assignment 8 you used the same dataset with 561 features that the authors computed.

**All the code is given.** You do not need to write code. Your work is to:
1. run the cells in order,
2. look at the results,
3. **explain** the results in the answer cells. Each answer cell has sentences for you to complete.

- **Part A (2.5 points):** train a CNN the honest way, and change the size of its kernels.
- **Part B (2.5 points):** CNN or features? Which sensors are important? How much data does a CNN need?

**Run time:** about 4 to 5 minutes in total on a CPU. A cell that trains a model shows `[*]` while it runs: wait until it is complete. Your numbers can be a little different from those of other students, because different computers give slightly different results. Use **your** numbers in your answers.

Total: 5 points.

## Words you need

| Word | Meaning |
| --- | --- |
| **window** | one piece of signal of 2.56 s (128 samples). Each window is one sample for the model. |
| **channel** | one signal, for example `total_acc_x`. Each window has 9 channels. |
| **epoch** | one pass of the training through all the training windows. |
| **loss** | the error that the training makes smaller. Lower is better. |
| **validation set** | subjects that we keep apart from the training, to choose the settings (for example the best epoch). |
| **test set** | subjects that we use **one time**, at the end, to measure the model on new people. |
| **early stopping** | keep the model from the epoch with the lowest validation loss. |
| **kernel size (receptive field)** | the number of samples that one kernel sees at the same time. |
| **recall** | of all the windows of one activity, the fraction that the model found (lecture 04.02). |

## Run this first

Run the cell below, at the start of every session. It downloads the course files to your Google Drive and keeps them up to date. When you see "Setup complete", you are ready to work.

**Do not edit or save your work inside the `AIforMovementScience` folder.** Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
try:
    from google.colab import drive
    os.chdir('/content')                  # start from a valid directory
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/AIforMovementScience'
    if os.path.isdir(REPO + '/.git'):
        # already downloaded before: just update it
        !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
    else:
        # first time, or a broken/partial copy: download a fresh copy
        !rm -rf "$REPO"
        !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"
except ImportError:
    # not in Colab: walk up from here until the repository root is found
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'lectures', 'data')):
        REPO = os.path.dirname(REPO)

WORKDIR = REPO + '/Assignments'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
DATA = REPO + '/lectures/data'        # the folder that holds the data files
print('Setup complete. You can now run the rest of the notebook.')

## How to submit

- Run all the cells in order (or use **Runtime > Run all**).
- Write your answers in the answer cells. Short answers are good: 1 or 2 sentences for each item.
- Download your notebook: **File > Download > Download .ipynb**, name it `YourName_Assignment_9.ipynb`, and upload it to Canvas.

---
# The data

Thirty volunteers did six activities with a smartphone on the waist: walking, walking upstairs, walking downstairs, sitting, standing and laying. The phone recorded at 50 samples/s. The signals were cut into **windows** of 2.56 s (128 samples). Each window has **9 channels**:

| Channels | Contents |
| --- | --- |
| `total_acc_x/y/z` | total acceleration, in g (**includes gravity**) |
| `body_acc_x/y/z` | body acceleration, in g (total acceleration **without gravity**) |
| `body_gyro_x/y/z` | angular velocity (rotation), in rad/s |

The authors split the data by subject: 21 subjects in the train set and 9 subjects in the test set.

Run the two cells below. The first cell loads the data (about 10 s). The second cell defines the models and the helper functions (the same CNN as in lecture 05.02c). You do not need to read all of the second cell.

In [ ]:
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'   # for Mac computers with Apple silicon
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

HAR = f"{DATA}/UCI_HAR_Dataset"
activities = ['Walking', 'Upstairs', 'Downstairs', 'Sitting', 'Standing', 'Laying']
channel_names = ['total_acc_x', 'total_acc_y', 'total_acc_z',
                 'body_acc_x',  'body_acc_y',  'body_acc_z',
                 'body_gyro_x', 'body_gyro_y', 'body_gyro_z']

def load_signals(part):
    """X: (windows, 9 channels, 128 samples), y: activity 0-5, subjects: subject of each window."""
    X = np.stack([np.loadtxt(f"{HAR}/{part}/Inertial Signals/{ch}_{part}.txt") for ch in channel_names], axis=1)
    y = np.loadtxt(f"{HAR}/{part}/y_{part}.txt").astype(int) - 1
    subjects = np.loadtxt(f"{HAR}/{part}/subject_{part}.txt").astype(int)
    return X.astype(np.float32), y, subjects

X_train_all, y_train_all, subj_train_all = load_signals('train')
X_test, y_test, subj_test = load_signals('test')

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('Data loaded. Device:', device)

In [ ]:
# ---------- Models ----------
class CNN1D(nn.Module):
    """The CNN of lecture 05.02c: two convolution blocks, then two fully connected layers."""
    def __init__(self, n_channels=9, kernel_size=3, n_outputs=6, n_timesteps=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size), nn.ReLU(), nn.Dropout(0.5), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size),         nn.ReLU(), nn.Dropout(0.5), nn.MaxPool1d(2),
        )
        with torch.no_grad():
            flat_dim = self.features(torch.zeros(1, n_channels, n_timesteps)).numel()
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(flat_dim, 100), nn.ReLU(), nn.Linear(100, n_outputs))

    def forward(self, x):
        return self.classifier(self.features(x))


class MultiHeadCNN(nn.Module):
    """Three CNN 'heads' in parallel, with kernels of 3, 7 and 15 samples. Their embeddings are joined."""
    def __init__(self, n_channels=9, kernel_sizes=(3, 7, 15), n_outputs=6, n_timesteps=128):
        super().__init__()
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(n_channels, 32, k), nn.ReLU(), nn.Dropout(0.5), nn.MaxPool1d(2),
                nn.Conv1d(32, 64, k),         nn.ReLU(), nn.Dropout(0.5), nn.MaxPool1d(2),
                nn.Flatten())
            for k in kernel_sizes])
        with torch.no_grad():
            flat_dim = sum(h(torch.zeros(1, n_channels, n_timesteps)).shape[1] for h in self.heads)
        self.classifier = nn.Sequential(nn.Linear(flat_dim, 100), nn.ReLU(), nn.Linear(100, n_outputs))

    def forward(self, x):
        return self.classifier(torch.cat([h(x) for h in self.heads], dim=1))


def new_cnn(n_channels=9, kernel_size=3, seed=0):
    """Make a new CNN1D. The seed fixes the random initial kernels."""
    torch.manual_seed(seed)
    return CNN1D(n_channels=n_channels, kernel_size=kernel_size).to(device)

def new_multihead(n_channels=9, seed=0):
    torch.manual_seed(seed)
    return MultiHeadCNN(n_channels=n_channels).to(device)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


# ---------- Training and prediction ----------
def train_cnn(model, X_tr, y_tr, X_val, y_val, epochs=15, lr=1e-3, seed=0, verbose=True):
    """Train with Adam and cross-entropy loss. After each epoch, compute the validation loss.
    At the end, the model is set back to the epoch with the LOWEST validation loss (early stopping).
    Returns: model, history (a DataFrame with one row per epoch), best_epoch."""
    torch.manual_seed(seed)
    loader = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)), batch_size=128, shuffle=True)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history, best_loss, best_state, best_epoch = [], np.inf, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, n = 0.0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(yb); n += len(yb)
        val_logits = predict_logits(model, X_val)
        val_loss = criterion(val_logits, torch.tensor(y_val)).item()
        val_acc = (val_logits.argmax(dim=1).numpy() == y_val).mean()
        history.append({'epoch': epoch, 'train_loss': total_loss / n, 'val_loss': val_loss, 'val_acc': val_acc})
        if val_loss < best_loss:
            best_loss, best_epoch, best_state = val_loss, epoch, copy.deepcopy(model.state_dict())
        if verbose:
            print(f"Epoch {epoch:2d} | train loss {total_loss / n:.3f} | validation loss {val_loss:.3f}, accuracy {val_acc:.3f}")
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch

@torch.no_grad()
def predict_logits(model, X, batch_size=512):
    model.eval()
    out = [model(torch.tensor(X[i:i + batch_size]).to(device)).cpu() for i in range(0, len(X), batch_size)]
    return torch.cat(out)

def predict(model, X):
    """Predicted activity (0-5) for each window."""
    return predict_logits(model, X).argmax(dim=1).numpy()

print('Helper functions ready.')

---
# Part A - Train a CNN the honest way (2.5 points)

## Question A1 (0.5 points) - Prepare the data

> **What this experiment does:** It computes the accuracy of a model that always predicts the most common activity (the **baseline**). Then it keeps 4 of the 21 train subjects as a **validation set**. Last, it scales each channel to mean 0 and standard deviation 1, with the mean and the standard deviation of the **training subjects only** (as in lecture 05.02c).

In [ ]:
print('Train:', X_train_all.shape, '| Test:', X_test.shape, '  (windows, channels, samples)')
print('Windows per activity (train):', dict(zip(activities, np.bincount(y_train_all).tolist())))

# baseline: always predict the most common activity of the train set
most_common = np.bincount(y_train_all).argmax()
baseline = (y_test == most_common).mean()
print(f'\nBaseline (always "{activities[most_common]}"): test accuracy = {baseline:.3f}')

# validation set: 4 of the 21 train subjects
rng = np.random.RandomState(0)
val_subjects = rng.choice(np.unique(subj_train_all), size=4, replace=False)
is_val = np.isin(subj_train_all, val_subjects)
X_tr, y_tr = X_train_all[~is_val], y_train_all[~is_val]
X_val, y_val = X_train_all[is_val], y_train_all[is_val]
print('\nValidation subjects:', np.sort(val_subjects))
print(f'Train: {len(y_tr)} windows from 17 subjects | Validation: {len(y_val)} windows from 4 subjects')

# scaling: one mean and one standard deviation per channel, from the TRAIN windows only
ch_mean = X_tr.mean(axis=(0, 2), keepdims=True)
ch_std = X_tr.std(axis=(0, 2), keepdims=True)
X_tr_s, X_val_s, X_test_s = [(X - ch_mean) / ch_std for X in (X_tr, X_val, X_test)]
print('\nChannel means after scaling (validation set):', X_val_s.mean(axis=(0, 2)).round(3))

> **What to look for:** the baseline accuracy, and the channel means of the validation set (they are close to 0, but not exactly 0).

**Your answer:** *(double-click this cell, and complete the sentences)*

*Example of the expected level (this first item is complete):*
1. A useful model must have a test accuracy higher than **0.18**, because a model that always says "Laying" already gets this accuracy without using the signals.

*Complete these:*

2. We compute the mean and the standard deviation from the training subjects only, because ...
3. We take the validation set from the train subjects, and not from the test subjects, because ...

## Question A2 (1 point) - Train and evaluate one CNN

> **What this experiment does:** It trains the CNN of lecture 05.02c (kernel size 3) for 20 epochs. After each epoch, it computes the loss on the train windows and on the validation windows. At the end, it keeps the model from the epoch with the lowest validation loss (**early stopping**). Then it uses the test set one time. This takes about 30 s.

In [ ]:
cnn3 = new_cnn(kernel_size=3)
print(f'Number of parameters: {count_parameters(cnn3):,}')
cnn3, hist3, best3 = train_cnn(cnn3, X_tr_s, y_tr, X_val_s, y_val, epochs=20)
print('\nBest epoch (lowest validation loss):', best3)

plt.figure(figsize=(7, 4))
plt.plot(hist3['epoch'], hist3['train_loss'], marker='o', label='Train loss')
plt.plot(hist3['epoch'], hist3['val_loss'], marker='o', label='Validation loss')
plt.axvline(best3, color='gray', linestyle='--', label='Best epoch')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Training history'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
pred3 = predict(cnn3, X_test_s)
val_acc3, test_acc3 = hist3.loc[best3 - 1, 'val_acc'], accuracy_score(y_test, pred3)
print(f"Validation accuracy (best epoch): {val_acc3:.3f}   (4 subjects)")
print(f"Test accuracy:                    {test_acc3:.3f}   (9 subjects)")

ConfusionMatrixDisplay.from_predictions(y_test, pred3, display_labels=activities, xticks_rotation=45, cmap='Blues')
plt.title('CNN (kernel 3), test set')
plt.show()

> **What to look for:** - **Loss plot:** does the validation loss go down, stay flat, or go up at the end? If it goes up while the train loss goes down, the model overfits.
- **Accuracies:** compare the validation and the test accuracy. Remember how many subjects are in each set.
- **Confusion matrix:** the large numbers outside the diagonal are the errors.

**Your answer:** *(double-click this cell, and complete the sentences)*

1. The best epoch is ___. After the first few epochs, the validation loss (goes down / stays about flat / goes up) ___, so the overfitting is (small / large) ___.
2. The validation accuracy is ___ and the test accuracy is ___. The (validation / test) ___ accuracy is higher. A reason is ...
3. The two activities that the CNN confuses most are ___ and ___. This makes sense for a phone on the waist, because ...

## Question A3 (1 point) - The size of the kernel (receptive field)

A small kernel sees fast, local changes. A large kernel sees slower patterns. At 50 samples/s, a kernel of 3 samples covers 0.06 s, and a kernel of 15 samples covers 0.30 s. One step of walking takes about 0.5 s.

The old version of this course used a **multi-head CNN**: three CNNs in parallel, with kernels of 3, 7 and 15 samples. The network then uses the results of the three.

> **What this experiment does:** It trains 4 models with the same data: a CNN with kernel size 3, 7 and 15, and the multi-head CNN. Each model trains for 10 epochs. The table shows the number of parameters, the validation accuracy and the test accuracy of each model. This takes about 2 minutes.

In [ ]:
rows = []
models_A3 = {'CNN, kernel 3':  lambda: new_cnn(kernel_size=3),
             'CNN, kernel 7':  lambda: new_cnn(kernel_size=7),
             'CNN, kernel 15': lambda: new_cnn(kernel_size=15),
             'Multi-head (3, 7, 15)': lambda: new_multihead()}
for name, make in models_A3.items():
    m = make()
    m, h, b = train_cnn(m, X_tr_s, y_tr, X_val_s, y_val, epochs=10, verbose=False)
    rows.append({'model': name, 'parameters': count_parameters(m),
                 'val accuracy': h.loc[b - 1, 'val_acc'], 'test accuracy': accuracy_score(y_test, predict(m, X_test_s))})
    print(name, 'done')
table_A3 = pd.DataFrame(rows).set_index('model')
table_A3.round(3)

> **What to look for:** the range of the test accuracies (the largest minus the smallest), and the number of parameters of the multi-head CNN.

**Your answer:** *(double-click this cell, and complete the sentences)*

1. The best test accuracy is from ___ (___). The range of the test accuracies is from ___ to ___.
2. The model with the most parameters is ___. It (is / is not) ___ the best model. This tells me that ...
3. With one training run and 9 test subjects, (I can / I cannot) ___ say that one kernel size is really better, because ... To be more sure, I would ...

---
# Part B - CNN or features? (2.5 points)

## Question B1 (0.75 points) - CNN vs. logistic regression on the 561 features

> **What this experiment does:** It trains the best model of Assignment 8 (logistic regression with the 561 features of the authors) on the **same 17 train subjects**. Then it compares this model with the CNN of A2 on the test set: accuracy, number of parameters, and confusion matrices side by side.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

F_train = pd.read_csv(f"{HAR}/train/X_train.txt", sep=r"\s+", header=None).values   # same windows, same order as the signals
F_test = pd.read_csv(f"{HAR}/test/X_test.txt", sep=r"\s+", header=None).values
lr = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000))])
lr.fit(F_train[~is_val], y_tr)
pred_lr = lr.predict(F_test)
test_acc_lr = accuracy_score(y_test, pred_lr)

print(f'Logistic regression, 561 features: test accuracy = {test_acc_lr:.3f} | parameters = {lr["model"].coef_.size + lr["model"].intercept_.size:,}')
print(f'CNN, raw signals:                  test accuracy = {test_acc3:.3f} | parameters = {count_parameters(cnn3):,}')

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay.from_predictions(y_test, pred_lr, display_labels=activities, xticks_rotation=45, cmap='Blues', ax=ax[0], colorbar=False)
ax[0].set_title('Logistic regression, 561 features')
ConfusionMatrixDisplay.from_predictions(y_test, pred3, display_labels=activities, xticks_rotation=45, cmap='Blues', ax=ax[1], colorbar=False)
ax[1].set_title('CNN, raw signals')
plt.tight_layout()
plt.show()

stand, sit = activities.index('Standing'), activities.index('Sitting')
print('Standing windows labeled as sitting: LR =', ((y_test == stand) & (pred_lr == sit)).sum(),
      '| CNN =', ((y_test == stand) & (pred3 == sit)).sum())

> **What to look for:** the two accuracies, and the number of standing windows that each model labels as sitting (row "Standing", column "Sitting").

**Your answer:** *(double-click this cell, and complete the sentences)*

1. The better model is ___ (test accuracy ___ vs. ___). The CNN labels ___ standing windows as sitting, and logistic regression labels ___.
2. The CNN has many more parameters, but it is not better here, because ... (Hint: what information do the 561 features contain? How many train subjects are there?)
3. A CNN on the raw signals can be the better choice when ...

## Question B2 (0.75 points) - Which sensors are important?

> **What this experiment does:** It trains the CNN two more times, each time with only some of the channels: (1) the total acceleration only, which includes gravity, and (2) the body acceleration and the gyroscope, which do **not** include gravity. The table compares them with the CNN of A2 (all 9 channels). For each activity, it shows the **recall**: the fraction of the windows of this activity that the model found. This takes about 30 s.

In [ ]:
from sklearn.metrics import recall_score

inputs_B2 = {'total acc only (3 ch, with gravity)': slice(0, 3),
             'body acc + gyro (6 ch, no gravity)': slice(3, 9)}
rows = []
for name, ch in inputs_B2.items():
    n_ch = len(range(9)[ch])
    m = new_cnn(n_channels=n_ch, kernel_size=3)
    m, h, b = train_cnn(m, X_tr_s[:, ch], y_tr, X_val_s[:, ch], y_val, epochs=10, verbose=False)
    pred = predict(m, X_test_s[:, ch])
    rows.append({'input': name, 'test accuracy': accuracy_score(y_test, pred),
                 **dict(zip(activities, recall_score(y_test, pred, average=None)))})
rows.append({'input': 'all 9 channels (A2)', 'test accuracy': test_acc3,
             **dict(zip(activities, recall_score(y_test, pred3, average=None)))})
table_B2 = pd.DataFrame(rows).set_index('input')
print('Columns after "test accuracy": recall of each activity')
table_B2.round(3)

> **What to look for:** the columns "Sitting", "Standing" and "Laying": the three postures with almost no movement. Which input gives low recalls for them?

**Your answer:** *(double-click this cell, and complete the sentences)*

1. The best input is ___. Without gravity (body acc + gyro), the recall of sitting goes from ___ to ___, and the recall of laying goes from ___ to ___.
2. This makes sense physically, because laying, sitting and standing have almost no movement, so ... (Hint: what is different between these postures, and which sensor measures it?)
3. In Assignment 8, the most important features of the random forest were ___. This (agrees / does not agree) ___ with B2, because ...

## Question B3 (0.5 points) - How much data does a CNN need?

> **What this experiment does:** It trains the CNN and the logistic regression with only 2, 5, 10 and 17 train subjects. The validation and the test sets do not change. The plot shows the test accuracy of the two models against the number of train subjects. This takes about 1 minute.

In [ ]:
X_train_all_s = (X_train_all - ch_mean) / ch_std
train_ids = np.random.RandomState(1).permutation(np.unique(subj_train_all[~is_val]))
n_list = [2, 5, 10, 17]
acc_cnn, acc_lr = [], []
for n in n_list:
    rows_n = np.isin(subj_train_all, train_ids[:n])          # windows of the first n subjects
    m = new_cnn()
    m, h, b = train_cnn(m, X_train_all_s[rows_n], y_train_all[rows_n], X_val_s, y_val, epochs=10, verbose=False)
    acc_cnn.append(accuracy_score(y_test, predict(m, X_test_s)))
    lr.fit(F_train[rows_n], y_train_all[rows_n])
    acc_lr.append(lr.score(F_test, y_test))
    print(f'{n:2d} subjects ({rows_n.sum():4d} windows): CNN {acc_cnn[-1]:.3f} | LR {acc_lr[-1]:.3f}')

plt.figure(figsize=(7, 4))
plt.plot(n_list, acc_cnn, marker='o', label='CNN, raw signals')
plt.plot(n_list, acc_lr, marker='s', label='Logistic regression, 561 features')
plt.xlabel('Number of train subjects'); plt.ylabel('Test accuracy'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

> **What to look for:** the gap between the two lines with 2 subjects and with 17 subjects. Each point is one training run, so small ups and downs are noise.

**Your answer:** *(double-click this cell, and complete the sentences)*

1. With 2 subjects, the CNN has ___ and logistic regression has ___. With 17 subjects, the CNN has ___ and logistic regression has ___.
2. The model that gains more from more subjects is ___, because ...

---
## Summary of your results

Run this cell. It collects your main results in one table. Use it for B4.

In [ ]:
summary = pd.DataFrame([
    {'experiment': 'Baseline (always the most common activity)', 'test accuracy': baseline},
    {'experiment': 'A2: CNN, kernel 3, all channels', 'test accuracy': test_acc3},
    *[{'experiment': f'A3: {k}', 'test accuracy': v} for k, v in table_A3['test accuracy'].items()],
    {'experiment': 'B1: logistic regression, 561 features', 'test accuracy': test_acc_lr},
    *[{'experiment': f'B2: CNN, {k}', 'test accuracy': v} for k, v in table_B2['test accuracy'].items() if 'A2' not in k],
    {'experiment': 'B3: CNN, 2 train subjects', 'test accuracy': acc_cnn[0]},
    {'experiment': 'B3: logistic regression, 2 train subjects', 'test accuracy': acc_lr[0]},
])
summary.set_index('experiment').round(3)

## Question B4 (0.5 points) - Plan for the final project

In the final project, you will answer one classification question with two types of input: **tabular gait parameters**, and the **full waveforms** with a 1D CNN.

**Your answer:** *(double-click this cell, and complete the sentences)*

1. Two rules from this assignment that I will use to evaluate my CNN in an honest way are: (a) ... and (b) ...
2. Before I train the models, I expect the (tabular model / CNN) ___ to be better, because ... (use your results from B1 and B3)
3. If the CNN is not better than the tabular model, the project (is / is not) ___ a failure, because ...

---
## Extra (no points)

If you want to try some code:
- In A2, change `epochs=20` to `epochs=60`. Does the validation loss start to go up?
- In A3, add `seed=1` to `new_cnn(...)` and to `train_cnn(...)`. Does the ranking of the models stay the same?